# DuckDB smoke test

This notebook opens the existing DuckDB database named by `DUCK_DB_SOURCE` in read-only mode. It tests SQL cells and bounded lazy collection without changing the database.

Install the project and set the source before starting Jupyter:

```bash
uv pip install -e '.[duckdb,viz]'
export DUCK_DB_SOURCE=/absolute/path/to/analytics.duckdb
jupyter lab examples/duckdb.ipynb
```

In [ ]:
import os

from sql_notebook_kit import create_session

duck_db_source = os.environ.get("DUCK_DB_SOURCE", "").strip()
if not duck_db_source:
    raise RuntimeError(
        "DUCK_DB_SOURCE must point to an existing DuckDB database before running this notebook"
    )

session = create_session(
    backend="duckdb",
    connection_kwargs={"database": duck_db_source, "read_only": True},
)
session.register(
    alias="duckdb_sample",
    login=True,
    visualization=True,
    max_rows=100,
)

Confirm which database DuckDB opened.

In [ ]:
%%sql
select current_database() as database_name, version() as duckdb_version

List up to 100 persistent tables through DuckDB's catalog. The notebook result is bounded by the session's `max_rows` setting.

In [ ]:
%%sql
select database_name, schema_name, table_name
from duckdb_tables()
order by database_name, schema_name, table_name
limit 100

Run the same catalog lookup through SQL Notebook Kit's separate lazy transformation connection.

In [ ]:
catalog_result = session.sql(
    """
    select database_name, schema_name, table_name
    from duckdb_tables()
    order by database_name, schema_name, table_name
    """
).collect(max_rows=100)
catalog_result

Dispose the eager and lazy connections when you finish testing.

In [ ]:
session.dispose()